# Manipulating DataFrame Values

**Learning Objectives:**
- Understand and avoid chained indexing warnings
- Safely modify DataFrame values using `.loc`
- Use modern pandas methods for value manipulation (`.clip()`, `.where()`, `.mask()`)
- Understand when to use `.copy()` vs. view references

[Link to pandas indexing documentation](https://pandas.pydata.org/docs/user_guide/indexing.html)

---

## 1. Setup and data loading

📚 **Concept**: Before we can manipulate data, we need to load it. We'll use a small Titanic dataset for practice.

<!-- INSTRUCTOR: Emphasize that this dataset is intentionally small for learning purposes -->

In [ ]:
import pandas as pd

In [ ]:
# Load our sample dataset
titanic = pd.read_csv("titanic_manipulate.csv")
titanic

---

## 2. Understanding chained indexing

📚 **Concept**: **Chained indexing** occurs when you use multiple indexing operations in a row. This can lead to unexpected behavior and warnings.

When modifying DataFrame values, pandas cannot always determine whether you're working with a copy or a view of the original data. This ambiguity causes the `SettingWithCopyWarning`.

<!-- INSTRUCTOR: Explain that this is one of the most common pandas mistakes -->

### 2.1 Example of chained indexing (problematic)

⚠️ **Warning**: The following code demonstrates a common mistake!

In [ ]:
# 🐛 INTENTIONAL ERROR - This will cause a warning!
titanic.loc[18]["age"] = 25
# This chains two operations: .loc[18] THEN ["age"]

🐛 **Debug**: The warning appears because:
1. `titanic.loc[18]` returns a Series (which might be a view or copy)
2. `["age"] = 25` tries to modify that Series
3. Pandas can't guarantee the original DataFrame will be modified

💡 **Tip**: If you see `SettingWithCopyWarning`, you're likely using chained indexing!

### 2.2 Correct approach: Use `.loc` with both row and column

✅ **Solution**: Specify both the row and column in a single `.loc` operation using a comma.

In [ ]:
# ✅ This works correctly - no warning!
titanic.loc[18, "age"] = 25

✅ **Key Takeaway**: Always use `.loc[row_indexer, col_indexer] = value` when setting values. Notice the **comma** between row and column selectors!

---

## 3. Modifying multiple values

📚 **Concept**: You can modify multiple values at once by passing lists to `.loc`.

<!-- INSTRUCTOR: Show how this scales from single to multiple values -->

In [ ]:
# Change age for rows 2 and 3
titanic.loc[[2, 3], "age"] = [13, 29]
titanic.head(5)

💡 **Tip**: The list of values `[13, 29]` must match the number of rows you're updating.

---

## 4. Filling missing values

📚 **Concept**: Missing values (NaN) are common in real-world data. We often need to replace them with meaningful values.

⚠️ **Modern Syntax**: Avoid using `inplace=True` parameter. Instead, reassign the result to the variable.

In [ ]:
# Reload data to start fresh
titanic = pd.read_csv("titanic_manipulate.csv")

In [ ]:
# Calculate mean age (ignoring NaN values)
mean_age = round(titanic.age.mean())
print(f"Mean age: {mean_age}")

In [ ]:
# Fill missing values with mean age
titanic = titanic.fillna(mean_age)
titanic.head()

💡 **Why avoid `inplace=True`?**
- Method chaining becomes easier
- Code is more explicit and readable
- Follows modern pandas best practices
- Future pandas versions may deprecate `inplace`

---

## 5. Conditional value modification

📚 **Concept**: Often we need to modify values based on a condition. For example: "Set all ages below 1 to 1."

Let's explore the **correct** and **modern** ways to do this.

<!-- INSTRUCTOR: This is where students often struggle - emphasize multiple valid approaches -->

In [ ]:
# Reload data to start fresh
titanic = pd.read_csv("titanic_manipulate.csv")

In [ ]:
# First, let's see which ages are below 1
titanic.loc[titanic["age"] < 1, "age"]

### 5.1 Wrong approach: Chained indexing

⚠️ **Warning**: This is a common mistake!

In [ ]:
# 🐛 INTENTIONAL ERROR - This will cause a warning!
titanic.loc[titanic["age"] < 1]["age"] = 1
# Problem: .loc[condition] THEN ["age"] = chained indexing

### 5.2 Correct approach: Use comma in `.loc`

In [ ]:
# Reload to fix the previous error
titanic = pd.read_csv("titanic_manipulate.csv")

In [ ]:
# ✅ Method 1: Use .loc with comma (row condition, column name)
titanic.loc[titanic["age"] < 1, "age"] = 1
titanic.loc[4:10, "age"]

✅ **Key Pattern**: `df.loc[condition, column] = value`

### 5.3 Modern approach: Use `.clip()`

📚 **Concept**: The `.clip()` method is a modern, efficient way to set minimum and maximum bounds on values.

In [ ]:
# Reload data
titanic = pd.read_csv("titanic_manipulate.csv")

In [ ]:
# ✅ Method 2: Use .clip() to set a minimum value
titanic["age"] = titanic["age"].clip(lower=1)
titanic.loc[4:10, "age"]

💡 **Advantages of `.clip()`**:
- Clear intent: "floor at 1"
- Works with both `lower` and `upper` bounds
- More efficient than conditional assignment
- No risk of chained indexing

<!-- INSTRUCTOR: Show .clip() with both lower and upper parameters -->

In [ ]:
# Example: Cap ages between 1 and 80
titanic["age"] = titanic["age"].clip(lower=1, upper=80)
titanic[["sex", "age"]]

### ✅ Quick Check

Before proceeding, can you:
- [ ] Explain why chained indexing causes warnings?
- [ ] Use `.loc[row, col] = value` correctly?
- [ ] Replace conditional assignment with `.clip()` where appropriate?

---

## 6. Advanced conditional replacement: `.where()` and `.mask()`

📚 **Concept**: For more complex conditional replacements, pandas provides `.where()` and `.mask()` methods.

- **`.where(condition, other)`**: Keep values WHERE condition is True, otherwise replace with `other`
- **`.mask(condition, other)`**: Replace values WHERE condition is True with `other`

<!-- INSTRUCTOR: These are inverse operations - emphasize the difference -->

In [ ]:
# Reload data
titanic = pd.read_csv("titanic_manipulate.csv")

In [ ]:
# Using .where(): Keep ages >= 1, replace others with 1
titanic["age"] = titanic["age"].where(titanic["age"] >= 1, other=1)
titanic.loc[4:10, "age"]

In [ ]:
# Reload for .mask() example
titanic = pd.read_csv("titanic_manipulate.csv")

In [ ]:
# Using .mask(): Replace ages < 1 with 1
titanic["age"] = titanic["age"].mask(titanic["age"] < 1, other=1)
titanic.loc[4:10, "age"]

💡 **When to use each**:
- **`.clip()`**: Setting min/max bounds (simplest)
- **`.loc[condition, col]`**: Replacing specific cells based on condition
- **`.where()` / `.mask()`**: Complex conditional replacement of entire columns

---

## 7. Understanding `.copy()` vs. view references

📚 **Concept**: When you create a new variable from a DataFrame, Python might create a **reference** (pointing to the same data) or a **copy** (independent data).

This distinction is crucial when you want to modify data without affecting the original.

<!-- INSTRUCTOR: Use diagrams or analogies - "two variables pointing to same notebook" vs "photocopying a page" -->

### 7.1 Without `.copy()` - Creating a reference

⚠️ **Warning**: This creates a reference, not an independent copy!

In [ ]:
# Start fresh
titanic = pd.read_csv("titanic_manipulate.csv")
titanic.head()

In [ ]:
# Create a "copy" without .copy() method
titanic_reference = titanic

In [ ]:
# Modify the "copy"
titanic_reference = titanic_reference.fillna(100)
titanic_reference.head()

In [ ]:
# ⚠️ Check the original - it's also changed!
titanic.head()

🐛 **Why did this happen?**

`titanic_reference = titanic` doesn't create a new DataFrame - it creates another name pointing to the same data.

Think of it like two people with keys to the same house. Changes made by one person affect what the other sees.

### 7.2 With `.copy()` - Creating an independent copy

✅ **Solution**: Use `.copy()` to create a truly independent DataFrame.

In [ ]:
# Reload original data
titanic = pd.read_csv("titanic_manipulate.csv")
titanic.head()

In [ ]:
# Create an independent copy
titanic_copy = titanic.copy()

In [ ]:
# Modify the copy
titanic_copy = titanic_copy.fillna(100)
titanic_copy.head()

In [ ]:
# ✅ Original remains unchanged!
titanic.head()

✅ **Key Takeaway**: Always use `.copy()` when you want to create an independent DataFrame that won't affect the original when modified.

### 7.3 When to use `.copy()`

💡 **Use `.copy()` when**:
- Creating a subset for separate analysis: `high_earners = df[df.salary > 100000].copy()`
- Testing transformations without affecting original data
- Creating multiple versions of data for comparison

💡 **You don't need `.copy()` when**:
- Just reading/viewing data
- Creating visualizations
- Performing calculations without assignment

---

## Summary

### 1. Avoiding Chained Indexing

❌ **Don't**: `df.loc[row]["col"] = value`  
✅ **Do**: `df.loc[row, "col"] = value`

**Rule**: Always use a comma in `.loc[row_indexer, col_indexer]` when setting values.

### 2. Modern Value Manipulation

| Task | Old Way | Modern Way |
|------|---------|------------|
| Fill missing | `df.fillna(0, inplace=True)` | `df = df.fillna(0)` |
| Floor values | `df.loc[df.age < 1, "age"] = 1` | `df["age"] = df["age"].clip(lower=1)` |
| Conditional replace | `df.loc[condition, col] = value` | `df[col] = df[col].where(condition, value)` |

### 3. Copy vs. Reference

- **Reference**: `df2 = df1` → Both variables point to same data
- **Copy**: `df2 = df1.copy()` → Independent DataFrames

✅ **Use `.copy()`** when creating subsets you'll modify independently.

### 4. Key Methods

📚 **Essential methods for value manipulation**:
- `.loc[row, col]` - Safe value assignment
- `.clip(lower, upper)` - Set min/max bounds
- `.where(condition, other)` - Keep values where condition is True
- `.mask(condition, other)` - Replace values where condition is True
- `.fillna(value)` - Replace missing values
- `.copy()` - Create independent copy

### ✅ Final Check

Can you now:
- [ ] Modify DataFrame values without chained indexing warnings?
- [ ] Use `.clip()` to set value bounds?
- [ ] Explain the difference between `.where()` and `.mask()`?
- [ ] Create independent DataFrame copies with `.copy()`?
- [ ] Choose between `.loc[]`, `.clip()`, and `.where()` for different tasks?

<!-- INSTRUCTOR: Review key patterns and preview lab exercises -->